This file does:
- Plotly store-location map
- Store counts by brand
- Migros versus competitor comparisons
- Population visualizations
- Merging store and population data
- Initial identification of underserved areas

In [177]:
import pandas as pd
import numpy as np
import plotly.express as px
import zipfile
from sklearn.neighbors import BallTree

stores_df = pd.read_csv(
    "data/switzerland_supermarkets_clean.csv"
)

population_df = pd.read_csv(
    "data/switzerland_population_clean.csv"
)

weight_df = pd.read_csv(
    "data/postal_code_municipality_mapping.csv"
)

taxpayers_df = pd.read_csv(
    "data/num_taxpayers_by_gemeinde.csv"
)

income_df = pd.read_csv(
    "data/total_income_by_gemeinde.csv"
)



#### EDA store_df

In [178]:
print("shape:", stores_df.shape)
stores_df.info()

shape: (2795, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2795 entries, 0 to 2794
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   osm_type       2795 non-null   object 
 1   osm_id         2795 non-null   int64  
 2   name           2795 non-null   object 
 3   brand          2601 non-null   object 
 4   operator       1162 non-null   object 
 5   shop_type      2795 non-null   object 
 6   street         1726 non-null   object 
 7   house_number   1693 non-null   object 
 8   postal_code    1634 non-null   float64
 9   city           1624 non-null   object 
 10  latitude       2795 non-null   float64
 11  longitude      2795 non-null   float64
 12  opening_hours  2156 non-null   object 
 13  website        1075 non-null   object 
 14  store_group    2795 non-null   object 
dtypes: float64(3), int64(1), object(11)
memory usage: 327.7+ KB


In [179]:
stores_df['postal_code'] = stores_df['postal_code'].astype('Int64')

In [180]:
missing = stores_df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


website          1720
operator         1633
city             1171
postal_code      1161
house_number     1102
street           1069
opening_hours     639
brand             194
dtype: int64

1,161 of 2,795 stores (41.53%) are missing postal_code. Only 1,634 stores can be used for postcode-level analysis.

In [181]:
stores_df[stores_df['postal_code'].isna()]['store_group'].value_counts()


store_group
Coop      432
Denner    399
Migros    197
Aldi       81
Lidl       52
Name: count, dtype: int64

Solve postal_code missing

In [182]:
with zipfile.ZipFile("data/ortschaftenverzeichnis_plz_4326.csv.zip") as z:
    with z.open("AMTOVZ_CSV_WGS84/AMTOVZ_CSV_WGS84.csv") as f:
        raw_locations = pd.read_csv(f, sep=";", encoding="utf-8-sig")

In [183]:
postcode_coordinates_df = (
    raw_locations[["PLZ4", "E", "N"]]
    .rename(columns={"PLZ4": "postal_code", "E": "postcode_longitude", "N": "postcode_latitude"})
    .drop_duplicates(subset="postal_code", keep="first")
    .reset_index(drop=True)
)

In [184]:
missing_mask = stores_df["postal_code"].isna()   # before fill

In [185]:
known = postcode_coordinates_df.dropna(subset=["postcode_latitude", "postcode_longitude"])
known_rad = np.radians(known[["postcode_latitude", "postcode_longitude"]].to_numpy())
tree = BallTree(known_rad, metric="haversine")

In [186]:
store_rad = np.radians(stores_df.loc[missing_mask, ["latitude", "longitude"]].to_numpy())
distance, index = tree.query(store_rad, k=1)


In [187]:
stores_df.loc[missing_mask, "postal_code"] = known.iloc[index.flatten()]["postal_code"].to_numpy()
stores_df.loc[missing_mask, "postcode_distance_km"] = distance.flatten() * 6371

stores_df["postcode_source"] = np.where(missing_mask, "estimated (nearest postcode)", "OSM address")

print(stores_df["postcode_source"].value_counts())
print("missing postal_code remaining:", stores_df["postal_code"].isna().sum())

postcode_source
OSM address                     1634
estimated (nearest postcode)    1161
Name: count, dtype: int64
missing postal_code remaining: 0


In [188]:
store_counts = stores_df["store_group"].value_counts().reset_index()
store_counts.columns = ["store_group", "count"]

fig = px.bar(
    store_counts,
    x="store_group",
    y="count",
    color="store_group",
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Number of locations by store group",
)
fig.update_layout(showlegend=False) 
fig.show()


In [189]:
color_map = {
    "Migros": "#E69F00",              # ส้มเด่น (สีแบรนด์ Migros) — สีนี้ผ่าน colorblind-safe check ด้วย
    "Coop": "#B0B0B0",
    "Denner": "#B0B0B0",
    "Aldi": "#B0B0B0",
    "Lidl": "#B0B0B0",
    "Migrolino": "#D0D0D0",
    "VOI Migros Partner": "#D0D0D0",
    "Other": "#E0E0E0",
}

fig = px.scatter_mapbox(
    stores_df,
    lat="latitude",
    lon="longitude",
    color="store_group",
    color_discrete_map=color_map,
    category_orders={"store_group": ["Migros", "Coop", "Denner", "Aldi", "Lidl", "Migrolino", "VOI Migros Partner", "Other"]},
    hover_name="name",
    hover_data=["city", "postal_code"],
    zoom=6.5,
    center={"lat": 46.8, "lon": 8.2},
    mapbox_style="open-street-map",
    title="Supermarket locations in Switzerland — Migros highlighted",
)
fig.update_layout(height=700)
fig.show()


C:\Users\ichay\AppData\Local\Temp\ipykernel_16768\3635842921.py:12: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [190]:
stores_by_postcode = pd.crosstab(stores_df["postal_code"], stores_df["store_group"])
stores_by_postcode["total_stores"] = stores_by_postcode.sum(axis=1)
stores_by_postcode = stores_by_postcode.reset_index()

stores_by_postcode.head()


store_group,postal_code,Aldi,Coop,Denner,Lidl,Migros,total_stores
0,1003,2,1,1,2,2,8
1,1004,0,2,1,0,3,6
2,1005,0,3,0,0,0,3
3,1006,0,2,2,0,1,5
4,1007,0,2,1,0,4,7


#### EDA population

In [191]:
print("shape:", population_df.shape)
population_df.info()
population_df.isna().sum().sort_values(ascending=False)


shape: (3176, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3176 entries, 0 to 3175
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   postal_code  3176 non-null   int64
 1   population   3176 non-null   int64
dtypes: int64(2)
memory usage: 49.8 KB


postal_code    0
population     0
dtype: int64

In [192]:
fig = px.histogram(
    population_df,
    x="population",
    nbins=50,
    color_discrete_sequence=["#0072B2"], 
    title="Distribution of population per postal code",
)
fig.update_layout(
    xaxis_title="Population",
    yaxis_title="Number of postal codes",
)
fig.show()


In [193]:
population_df["population"].describe()


count     3176.000000
mean      2873.779912
std       4817.799564
min          1.000000
25%        494.750000
50%       1159.500000
75%       2981.500000
max      42944.000000
Name: population, dtype: float64

#### EDA taxpayers_df

In [194]:
print("shape:", taxpayers_df.shape)
taxpayers_df.info()

shape: (2148, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2148 entries, 0 to 2147
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Kanton ID                 2148 non-null   float64
 1   Kanton / Canton           2148 non-null   object 
 2   Gemeinde ID               2148 non-null   int64  
 3   Gemeinde / Commune        2148 non-null   object 
 4   0                         2148 non-null   float64
 5   1 - 30'000                2148 non-null   float64
 6   30'001 - 40'000           2148 non-null   float64
 7   40'001 - 50'000           2148 non-null   float64
 8   50'001 - 75'000           2148 non-null   float64
 9   75'001 - 100'000          2148 non-null   float64
 10  100'001 - 200'000         2148 non-null   float64
 11  200'001 - 500'000         2148 non-null   float64
 12  500'001 - 1'000'000       2148 non-null   float64
 13  1'000'001 - u.m./et plus  2148 non-null   flo

In [195]:
taxpayers_df.head()

,Kanton ID,Kanton / Canton,Gemeinde ID,Gemeinde / Commune,0,1 - 30'000,30'001 - 40'000,40'001 - 50'000,50'001 - 75'000,75'001 - 100'000,100'001 - 200'000,200'001 - 500'000,500'001 - 1'000'000,1'000'001 - u.m./et plus,num_taxpayers
0,1.0,Zürich,1,Aeugst am Albis,0.0,222.0,73.0,78.0,179.0,139.0,284.0,96.0,12.0,0.0,1087
1,1.0,Zürich,2,Affoltern am Albis,0.0,1540.0,540.0,725.0,1692.0,982.0,1126.0,210.0,11.0,0.0,6829
2,1.0,Zürich,3,Bonstetten,0.0,541.0,161.0,206.0,568.0,488.0,754.0,188.0,7.0,0.0,2914
3,1.0,Zürich,4,Hausen am Albis,4.0,429.0,132.0,213.0,440.0,285.0,457.0,117.0,6.0,0.0,2086
4,1.0,Zürich,5,Hedingen,0.0,402.0,126.0,160.0,425.0,322.0,473.0,135.0,17.0,0.0,2064


In [196]:
taxpayers_df.isna().sum()


Kanton ID                   0
Kanton / Canton             0
Gemeinde ID                 0
Gemeinde / Commune          0
0                           0
1 - 30'000                  0
30'001 - 40'000             0
40'001 - 50'000             0
50'001 - 75'000             0
75'001 - 100'000            0
100'001 - 200'000           0
200'001 - 500'000           0
500'001 - 1'000'000         0
1'000'001 - u.m./et plus    0
num_taxpayers               0
dtype: int64

In [197]:
taxpayers_df["num_taxpayers"].describe()

count      2148.000000
mean       2233.921322
std        7236.639322
min          18.000000
25%         419.750000
50%         908.000000
75%        2149.500000
max      237310.000000
Name: num_taxpayers, dtype: float64

In [198]:
fig = px.histogram(
    taxpayers_df,
    x="num_taxpayers",
    nbins=50,
    color_discrete_sequence=["#0072B2"],
    title="Distribution of taxpayers per municipality",
)
fig.update_layout(xaxis_title="Number of taxpayers", yaxis_title="Number of municipalities")
fig.show()


#### EDA Income

In [199]:
print("shape:", income_df.shape)
income_df.info()

shape: (2148, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2148 entries, 0 to 2147
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Kanton ID                 2148 non-null   float64
 1   Kanton / Canton           2148 non-null   object 
 2   Gemeinde ID               2148 non-null   int64  
 3   Gemeinde / Commune        2148 non-null   object 
 4   0                         2148 non-null   float64
 5   1 - 30'000                2148 non-null   float64
 6   30'001 - 40'000           2148 non-null   float64
 7   40'001 - 50'000           2148 non-null   float64
 8   50'001 - 75'000           2148 non-null   float64
 9   75'001 - 100'000          2148 non-null   float64
 10  100'001 - 200'000         2148 non-null   float64
 11  200'001 - 500'000         2148 non-null   float64
 12  500'001 - 1'000'000       2148 non-null   float64
 13  1'000'001 - u.m./et plus  2148 non-null   flo

In [200]:
income_df.isna().sum()


Kanton ID                   0
Kanton / Canton             0
Gemeinde ID                 0
Gemeinde / Commune          0
0                           0
1 - 30'000                  0
30'001 - 40'000             0
40'001 - 50'000             0
50'001 - 75'000             0
75'001 - 100'000            0
100'001 - 200'000           0
200'001 - 500'000           0
500'001 - 1'000'000         0
1'000'001 - u.m./et plus    0
Total_income                0
dtype: int64

In [201]:
income_df["Total_income"].describe()


count    2.148000e+03
mean     1.547376e+08
std      5.147586e+08
min      6.914580e+05
25%      2.773144e+07
50%      6.297404e+07
75%      1.489599e+08
max      1.830524e+10
Name: Total_income, dtype: float64

In [202]:
fig = px.histogram(
    income_df,
    x="Total_income",
    nbins=50,
    color_discrete_sequence=["#0072B2"],
    title="Distribution of total income per municipality",
)
fig.update_layout(xaxis_title="Total income (CHF)", yaxis_title="Number of municipalities")
fig.show()


#### Merge postcode and tax data

In [203]:
postcode_tax = weight_df.merge(
    taxpayers_df[["Gemeinde ID", "num_taxpayers"]],
    left_on="municipality_id", right_on="Gemeinde ID", how="left"
).merge(
    income_df[["Gemeinde ID", "Total_income"]],
    left_on="municipality_id", right_on="Gemeinde ID", how="left"
)

postcode_tax.head(10)

,postal_code,municipality_id,weight_pct,municipality_name,canton_code,Gemeinde ID_x,num_taxpayers,Gemeinde ID_y,Total_income
0,1000,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09
1,1003,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09
2,1004,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09
3,1005,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09
4,1006,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09
5,1007,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09
6,1008,5585,50.0,Jouxtens-Mézery,VD,5585.0,722.0,5585.0,1.073640e+08
7,1008,5589,50.0,Prilly,VD,5589.0,6525.0,5589.0,3.629417e+08
8,1009,5590,100.0,Pully,VD,5590.0,9790.0,5590.0,8.451438e+08
9,1010,5586,100.0,Lausanne,VD,5586.0,77121.0,5586.0,4.410409e+09


**⚠️ Bug:** same municipality total (e.g. Lausanne 77121) was copied onto every postcode
it covers, instead of being split among them. Fixed below using population share
(`share_of_muni`) instead of `weight_pct` — verified: Lausanne's postcodes now sum back
to exactly 77,121.


In [204]:
pm = weight_df.merge(population_df, on="postal_code")
pm.head()

,postal_code,municipality_id,weight_pct,municipality_name,canton_code,population
0,1000,5586,100.0,Lausanne,VD,4248
1,1003,5586,100.0,Lausanne,VD,6879
2,1004,5586,100.0,Lausanne,VD,31463
3,1005,5586,100.0,Lausanne,VD,12454
4,1006,5586,100.0,Lausanne,VD,15621


In [205]:
pm["pop_share"] = pm["population"] * pm["weight_pct"] / 100
pm.head()

,postal_code,municipality_id,weight_pct,municipality_name,canton_code,population,pop_share
0,1000,5586,100.0,Lausanne,VD,4248,4248.0
1,1003,5586,100.0,Lausanne,VD,6879,6879.0
2,1004,5586,100.0,Lausanne,VD,31463,31463.0
3,1005,5586,100.0,Lausanne,VD,12454,12454.0
4,1006,5586,100.0,Lausanne,VD,15621,15621.0


In [206]:
muni_pop = pm.groupby("municipality_id")["pop_share"].sum().reset_index(name="muni_total_pop")
muni_pop.head()


,municipality_id,muni_total_pop
0,1,2003.00000
1,2,12880.93019
2,3,5659.00000
3,4,11103.00000
4,5,3983.00000


In [207]:
pm = pm.merge(muni_pop, on="municipality_id")
pm["share_of_muni"] = pm["pop_share"] / pm["muni_total_pop"]

pm[pm["municipality_id"] == 5586]  # check Lausanne


,postal_code,municipality_id,weight_pct,municipality_name,canton_code,population,pop_share,muni_total_pop,share_of_muni
0,1000,5586,100.000,Lausanne,VD,4248,4248.00000,146524.70197,0.028992
1,1003,5586,100.000,Lausanne,VD,6879,6879.00000,146524.70197,0.046948
2,1004,5586,100.000,Lausanne,VD,31463,31463.00000,146524.70197,0.214728
3,1005,5586,100.000,Lausanne,VD,12454,12454.00000,146524.70197,0.084996
4,1006,5586,100.000,Lausanne,VD,15621,15621.00000,146524.70197,0.106610
5,1007,5586,100.000,Lausanne,VD,22716,22716.00000,146524.70197,0.155032
9,1010,5586,100.000,Lausanne,VD,15745,15745.00000,146524.70197,0.107456
10,1011,5586,100.000,Lausanne,VD,55,55.00000,146524.70197,0.000375
11,1012,5586,100.000,Lausanne,VD,12140,12140.00000,146524.70197,0.082853
12,1018,5586,100.000,Lausanne,VD,23692,23692.00000,146524.70197,0.161693


In [208]:
pm = pm.merge(taxpayers_df[["Gemeinde ID", "num_taxpayers"]], left_on="municipality_id", right_on="Gemeinde ID")
pm = pm.merge(income_df[["Gemeinde ID", "Total_income"]], left_on="municipality_id", right_on="Gemeinde ID")
pm.head(10)

,postal_code,municipality_id,weight_pct,municipality_name,canton_code,population,pop_share,muni_total_pop,share_of_muni,Gemeinde ID_x,num_taxpayers,Gemeinde ID_y,Total_income
0,1000,5586,100.0,Lausanne,VD,4248,4248.0,146524.70197,0.028992,5586,77121,5586,4410409326
1,1003,5586,100.0,Lausanne,VD,6879,6879.0,146524.70197,0.046948,5586,77121,5586,4410409326
2,1004,5586,100.0,Lausanne,VD,31463,31463.0,146524.70197,0.214728,5586,77121,5586,4410409326
3,1005,5586,100.0,Lausanne,VD,12454,12454.0,146524.70197,0.084996,5586,77121,5586,4410409326
4,1006,5586,100.0,Lausanne,VD,15621,15621.0,146524.70197,0.106610,5586,77121,5586,4410409326
5,1007,5586,100.0,Lausanne,VD,22716,22716.0,146524.70197,0.155032,5586,77121,5586,4410409326
6,1008,5585,50.0,Jouxtens-Mézery,VD,14475,7237.5,7237.50000,1.000000,5585,722,5585,107364020
7,1008,5589,50.0,Prilly,VD,14475,7237.5,7237.50000,1.000000,5589,6525,5589,362941702
8,1009,5590,100.0,Pully,VD,19396,19396.0,19517.00000,0.993800,5590,9790,5590,845143768
9,1010,5586,100.0,Lausanne,VD,15745,15745.0,146524.70197,0.107456,5586,77121,5586,4410409326


In [209]:
# Calculate number of taxpyer and income according to population structure
pm["weighted_taxpayers"] = pm["num_taxpayers"] * pm["share_of_muni"]
pm["weighted_income"] = pm["Total_income"] * pm["share_of_muni"]

pm[pm["municipality_id"] == 5586][["postal_code", "share_of_muni", "weighted_taxpayers", "weighted_income"]]


,postal_code,share_of_muni,weighted_taxpayers,weighted_income
0,1000,0.028992,2235.868789,1.278653e+08
1,1003,0.046948,3620.654756,2.070586e+08
2,1004,0.214728,16560.061139,9.470397e+08
3,1005,0.084996,6554.969374,3.748667e+08
4,1006,0.106610,8221.870612,4.701938e+08
5,1007,0.155032,11956.213611,6.837540e+08
9,1010,0.107456,8287.136085,4.739262e+08
10,1011,0.000375,28.948395,1.655506e+06
11,1012,0.082853,6389.700354,3.654153e+08
12,1018,0.161693,12469.916044,7.131318e+08


In [210]:
postcode_income_summary = pm.groupby("postal_code")[["weighted_taxpayers", "weighted_income"]].sum().reset_index()

postcode_income_summary.head(10)


,postal_code,weighted_taxpayers,weighted_income
0,1000,2235.868789,1.278653e+08
1,1003,3620.654756,2.070586e+08
2,1004,16560.061139,9.470397e+08
3,1005,6554.969374,3.748667e+08
4,1006,8221.870612,4.701938e+08
5,1007,11956.213611,6.837540e+08
6,1008,7247.000000,4.703057e+08
7,1009,9729.304709,8.399041e+08
8,1010,8287.136085,4.739262e+08
9,1011,28.948395,1.655506e+06


In [211]:
postcode_income_summary["weighted_taxpayers"] = postcode_income_summary["weighted_taxpayers"].round(0).astype(int)
postcode_income_summary

,postal_code,weighted_taxpayers,weighted_income
0,1000,2236,1.278653e+08
1,1003,3621,2.070586e+08
2,1004,16560,9.470397e+08
3,1005,6555,3.748667e+08
4,1006,8222,4.701938e+08
...,...,...,...
3141,9652,407,2.346194e+07
3142,9655,189,1.090485e+07
3143,9656,362,2.021631e+07
3144,9657,389,2.173633e+07


#### Merged everything

In [212]:
# population on left because it may be the case that some postal codes have no store 

merged_df = population_df.merge(stores_by_postcode, on="postal_code", how="outer")
store_cols = list(stores_by_postcode.columns[1:])

merged_df[store_cols] = merged_df[store_cols].fillna(0)
merged_df["population"] = merged_df["population"].fillna(0)

merged_df = merged_df.merge(postcode_income_summary, on="postal_code", how="left")
merged_df


,postal_code,population,Aldi,Coop,Denner,Lidl,Migros,total_stores,weighted_taxpayers,weighted_income
0,1000,4248.0,0.0,0.0,0.0,0.0,0.0,0.0,2236.0,1.278653e+08
1,1003,6879.0,2.0,1.0,1.0,2.0,2.0,8.0,3621.0,2.070586e+08
2,1004,31463.0,0.0,2.0,1.0,0.0,3.0,6.0,16560.0,9.470397e+08
3,1005,12454.0,0.0,3.0,0.0,0.0,0.0,3.0,6555.0,3.748667e+08
4,1006,15621.0,0.0,2.0,2.0,0.0,1.0,5.0,8222.0,4.701938e+08
...,...,...,...,...,...,...,...,...,...,...
3182,9652,781.0,0.0,0.0,0.0,0.0,1.0,1.0,407.0,2.346194e+07
3183,9655,363.0,0.0,0.0,0.0,0.0,0.0,0.0,189.0,1.090485e+07
3184,9656,665.0,0.0,0.0,0.0,0.0,0.0,0.0,362.0,2.021631e+07
3185,9657,715.0,0.0,1.0,0.0,0.0,0.0,1.0,389.0,2.173633e+07


In [213]:
fill_cols = list(stores_by_postcode.columns[1:]) + ["weighted_taxpayers", "weighted_income"]
merged_df[fill_cols] = merged_df[fill_cols].fillna(0)
merged_df

,postal_code,population,Aldi,Coop,Denner,Lidl,Migros,total_stores,weighted_taxpayers,weighted_income
0,1000,4248.0,0.0,0.0,0.0,0.0,0.0,0.0,2236.0,1.278653e+08
1,1003,6879.0,2.0,1.0,1.0,2.0,2.0,8.0,3621.0,2.070586e+08
2,1004,31463.0,0.0,2.0,1.0,0.0,3.0,6.0,16560.0,9.470397e+08
3,1005,12454.0,0.0,3.0,0.0,0.0,0.0,3.0,6555.0,3.748667e+08
4,1006,15621.0,0.0,2.0,2.0,0.0,1.0,5.0,8222.0,4.701938e+08
...,...,...,...,...,...,...,...,...,...,...
3182,9652,781.0,0.0,0.0,0.0,0.0,1.0,1.0,407.0,2.346194e+07
3183,9655,363.0,0.0,0.0,0.0,0.0,0.0,0.0,189.0,1.090485e+07
3184,9656,665.0,0.0,0.0,0.0,0.0,0.0,0.0,362.0,2.021631e+07
3185,9657,715.0,0.0,1.0,0.0,0.0,0.0,1.0,389.0,2.173633e+07


In [214]:
merged_df["competitor_count"] = (
    merged_df["Aldi"]
    + merged_df["Coop"]
    + merged_df["Denner"]
    + merged_df["Lidl"]
)

print("shape:", merged_df.shape)
merged_df.head()

shape: (3187, 11)


,postal_code,population,Aldi,Coop,Denner,Lidl,Migros,total_stores,weighted_taxpayers,weighted_income,competitor_count
0,1000,4248.0,0.0,0.0,0.0,0.0,0.0,0.0,2236.0,1.278653e+08,0.0
1,1003,6879.0,2.0,1.0,1.0,2.0,2.0,8.0,3621.0,2.070586e+08,6.0
2,1004,31463.0,0.0,2.0,1.0,0.0,3.0,6.0,16560.0,9.470397e+08,3.0
3,1005,12454.0,0.0,3.0,0.0,0.0,0.0,3.0,6555.0,3.748667e+08,3.0
4,1006,15621.0,0.0,2.0,2.0,0.0,1.0,5.0,8222.0,4.701938e+08,4.0


In [215]:

fig = px.scatter(
    merged_df,
    x="competitor_count",
    y="Migros",
    size="population",
    title="Migros presence vs competitor presence per postcode",
    color_discrete_sequence=["#E69F00"],
)
fig.show()


In [216]:
total = len(merged_df)
no_store_at_all = ((merged_df["Migros"] == 0) & (merged_df["competitor_count"] == 0)).sum()
gap_area = ((merged_df["Migros"] == 0) & (merged_df["competitor_count"] > 0)).sum()
has_migros = (merged_df["Migros"] > 0).sum()

summary = pd.DataFrame({
    "group": ["No store at all", "Competitor present, no Migros (gap)", "Migros present"],
    "postcode_count": [no_store_at_all, gap_area, has_migros],
    "pct": [no_store_at_all/total*100, gap_area/total*100, has_migros/total*100],
})
print("total postcodes:", total)
summary


total postcodes: 3187


,group,postcode_count,pct
0,No store at all,2227,69.877628
1,"Competitor present, no Migros (gap)",410,12.864763
2,Migros present,550,17.257609


In [217]:
# Export
merged_df.to_csv("data/migros_for_analysis.csv", index=False)
print("saved:", len(merged_df), "rows -> data/migros_for_analysis.csv")


saved: 3187 rows -> data/migros_for_analysis.csv
